# Chaos in practice — exercise lab

This laboratory accompanies [the chapter](https://mvreeuwijk.github.io/chaosbook/python/practice.html). It runs
entirely in your browser (Python via Pyodide) — nothing to install, and any
changes you make are yours alone: re-opening the lab from the chapter resets it.

The full text of the exercises is in the
[chapter's exercise section](https://mvreeuwijk.github.io/chaosbook/python/practice.html#exercises); this notebook
gives you a running start on each one. The data files for the last exercise are
in this lab's filesystem under `data/`.

In [ ]:
# Run this cell first: it installs the book's package into the
# in-browser Python (takes a few seconds, needs to run once per visit).
%pip install -q chaosbook ipywidgets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from ipywidgets import interact, FloatSlider, IntSlider, Checkbox
import chaosbook as cb

def lorenz_ensemble(x0, eps0, M, t, seed=0):
    """Integrate M copies of x0 perturbed by eps0 (as one system of 3M ODEs).
    Returns an array of shape (len(t), M, 3)."""
    rng = np.random.default_rng(seed)
    ics = x0 + eps0 * rng.standard_normal((M, 3))
    f = lambda s, Y: np.ravel(cb.lorenz(s, Y.reshape(3, M)))
    sol = solve_ivp(f, [0, t[-1]], ics.T.ravel(), t_eval=t, rtol=1e-9, atol=1e-11)
    return sol.y.reshape(3, M, -1).transpose(2, 1, 0)

def spread(cloud):
    """RMS distance of the members from the ensemble mean, at each time."""
    d = cloud - cloud.mean(axis=1, keepdims=True)
    return np.sqrt((d ** 2).sum(axis=2).mean(axis=1))

# a point on the attractor: integrate away a transient
x0 = solve_ivp(cb.lorenz, [0, 50], [1, 1, 1], rtol=1e-10, atol=1e-12).y[:, -1]

## Explorer: ensemble spread versus initial uncertainty

Nothing is precomputed here. Choose the initial spread $\delta_0 = 10^{\,\log_{10}\delta_0}$
and the ensemble size, and watch the spread $\delta(t)$ grow as roughly
$\mathrm{e}^{\Lambda_e t}$ (dashed, $\Lambda_e = 0.9$) until it saturates at the size of the
attractor. The vertical line marks when the spread first reaches 40% of its
saturation value: shrink $\delta_0$ by a factor of $10^{3}$ and see how little
extra forecast time it buys.

In [ ]:
def explore(log10_delta0=-3.0, M=200, tend=22.0, Le=0.9):
    t = np.linspace(0, tend, 5 * int(tend) * 10 + 1)
    delta = spread(lorenz_ensemble(x0, 10.0**log10_delta0, M, t))
    sat = delta[t >= 0.8 * tend].mean()
    horizon = t[np.argmax(delta > 0.4 * sat)]
    plt.figure(figsize=(7, 3.5))
    plt.semilogy(t, delta, label="ensemble spread")
    plt.semilogy(t, delta[0] * np.exp(Le * t), "k--", label=r"$\propto e^{\Lambda_e t}$")
    plt.axhline(sat, color="r", ls=":", label="attractor size")
    plt.axvline(horizon, color="orange")
    plt.ylim(1e-6, 1e2)
    plt.xlabel("$t$")
    plt.ylabel(r"$\delta(t)$")
    plt.title(f"horizon (40% of saturation) at t = {horizon:.1f}")
    plt.legend(loc="lower right", frameon=False)
    plt.show()

interact(explore,
         log10_delta0=FloatSlider(-3.0, min=-8, max=0, step=0.5),
         M=IntSlider(200, min=20, max=600, step=20),
         tend=FloatSlider(22.0, min=10, max=40, step=2),
         Le=FloatSlider(0.9, min=0.3, max=1.5, step=0.05));

## Exercise: the predictability horizon

Full text: [parts a)–c) in the chapter](https://mvreeuwijk.github.io/chaosbook/python/practice.html#exercises).
The chapter's ensemble code is prefilled below.

In [ ]:
# Part a): M = 400 members from a Gaussian blob of width eps0 around a point
# on the attractor. Is the growth rate close to Lambda_e = 0.9?
M, eps0 = 400, 1e-3
t = np.linspace(0, 22, 2201)
delta = spread(lorenz_ensemble(x0, eps0, M, t))

window = (t > 1) & (t < 6)                       # a stretch of clean exponential growth
rate = np.polyfit(t[window], np.log(delta[window]), 1)[0]
print(f"fitted growth rate: {rate:.2f}")
plt.semilogy(t, delta)
plt.semilogy(t, delta[0] * np.exp(0.9 * t), "k--")
plt.xlabel("$t$")
plt.ylabel(r"$\delta(t)$")
plt.ylim(1e-4, 1e2)
plt.show()

In [ ]:
# Parts b)-c): halve eps0 and compare the time at which the spread first
# reaches 40% of its saturation level. Where does the growth stop (part c)?
def horizon(eps0, M=400):
    d = spread(lorenz_ensemble(x0, eps0, M, t))
    sat = d[t >= 18].mean()
    return t[np.argmax(d > 0.4 * sat)], sat

for e in (1e-3, 5e-4):
    th, sat = horizon(e)
    print(f"eps0 = {e:g}: horizon {th:.2f}, saturation level {sat:.1f}")

## Exercise: a Kalman filter for the Lorenz system

Full text: [parts a)–c) in the chapter](https://mvreeuwijk.github.io/chaosbook/python/practice.html#exercises).
The twin experiment is set up for you with the chapter's extended Kalman
filter; the observation interval and noise level are parameters, and the
`observed` mask selects which components are measured (part c).

In [ ]:
sigma, r, b = 10.0, 28.0, 8 / 3

def jac(x, y, z):                                  # Jacobian of the Lorenz equations
    return np.array([[-sigma, sigma, 0], [r - z, -1, -x], [y, x, -b]])

def forecast(xa, dtobs):
    """Advance the state and solve dPhi/dt = J Phi over one window."""
    def rhs(t, s):
        Phi = s[3:].reshape(3, 3)
        return np.concatenate([cb.lorenz(t, s[:3]), (jac(*s[:3]) @ Phi).ravel()])
    s0 = np.concatenate([xa, np.eye(3).ravel()])
    s = solve_ivp(rhs, [0, dtobs], s0, rtol=1e-9, atol=1e-11).y[:, -1]
    return s[:3], s[3:].reshape(3, 3)

def twin_experiment(dtobs=0.2, sig_obs=1.5, T=12.0, observed=(True, True, True), seed=7):
    Nobs = int(round(T / dtobs))
    rng = np.random.default_rng(seed)
    tobs = dtobs * np.arange(1, Nobs + 1)
    truth = solve_ivp(cb.lorenz, [0, T], x0, t_eval=tobs, rtol=1e-9, atol=1e-11).y.T
    y = truth + sig_obs * rng.standard_normal((Nobs, 3))
    H = np.eye(3)[list(observed)]                  # which components are measured
    R = sig_obs**2 * np.eye(H.shape[0])
    Q = 0.01 * np.eye(3)
    xa, Pa = x0 + 8.0, 4.0 * np.eye(3)             # wrong first guess and its uncertainty
    free = x0 + 8.0
    Xa, Xf = [], []
    for k in range(Nobs):
        xf, Phi = forecast(xa, dtobs)
        Pf = 1.03 * Phi @ Pa @ Phi.T + Q           # (mild inflation)
        K = Pf @ H.T @ np.linalg.inv(H @ Pf @ H.T + R)
        xa = xf + K @ (y[k][list(observed)] - H @ xf)
        Pa = (np.eye(3) - K @ H) @ Pf
        free = forecast(free, dtobs)[0]            # the free run never sees the data
        Xa.append(xa); Xf.append(free)
    return tobs, truth, y, np.array(Xa), np.array(Xf)

def show(dtobs=0.2, sig_obs=1.5, x_only=False):
    obs = (True, False, False) if x_only else (True, True, True)
    tobs, truth, y, Xa, Xf = twin_experiment(dtobs, sig_obs, observed=obs)
    rms = lambda A: np.sqrt(((A - truth) ** 2).mean(axis=1))
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
    a1.plot(tobs, truth[:, 0], "k", label="truth")
    a1.plot(tobs, Xf[:, 0], "r--", label="free forecast")
    a1.plot(tobs, Xa[:, 0], "b", label="analysis")
    a1.plot(tobs, y[:, 0], ".", color="orange", label="observations")
    a1.set_ylabel("$x$")
    a1.legend(ncol=4, frameon=False, loc="upper center")
    a2.semilogy(tobs, rms(Xf), "r--")
    a2.semilogy(tobs, rms(Xa), "b")
    a2.set_ylabel("RMS error")
    a2.set_xlabel("$t$")
    plt.show()

interact(show,
         dtobs=FloatSlider(0.2, min=0.05, max=1.0, step=0.05),
         sig_obs=FloatSlider(1.5, min=0.1, max=5.0, step=0.1),
         x_only=Checkbox(False, description="observe x only"));

## Exercise: reduced-order structure

Full text: [parts a)–c) in the chapter](https://mvreeuwijk.github.io/chaosbook/python/practice.html#exercises).
Parts a)–c) start from the synthetic field below: three fixed spatial patterns
whose amplitudes are the components of a Lorenz trajectory.

In [ ]:
# Part a): the snapshot matrix (columns are snapshots) and its SVD.
Nx, Nt = 200, 800
xg = np.linspace(0, 2 * np.pi, Nx)
phi = [np.sin(xg), np.sin(2 * xg) * np.exp(-((xg - np.pi) ** 2) / 4), np.cos(3 * xg)]
traj = solve_ivp(cb.lorenz, [0, 8], x0, t_eval=np.arange(Nt) * 0.01,
                 rtol=1e-9, atol=1e-11).y.T
a = (traj - traj.mean(0)) / traj.std(0)            # normalised amplitudes a_k(t)
noise = 0.15                                       # try 0, 0.15, 0.5
rng = np.random.default_rng(11)
field = sum(np.outer(a[:, k], phi[k]) for k in range(3)) + noise * rng.standard_normal((Nt, Nx))
Xm = field.T - field.T.mean(axis=1, keepdims=True)
U, S, Vt = np.linalg.svd(Xm, full_matrices=False)

In [ ]:
# Part b): the singular values - how many modes capture 99% of the energy?
energy = S**2 / np.sum(S**2)
# (the noise spreads a little energy over very many modes: also try noise = 0, or 95%)
print("modes for 99% of the energy:", np.searchsorted(np.cumsum(energy), 0.99) + 1)
plt.bar(range(1, 11), energy[:10])
plt.xlabel("mode number")
plt.ylabel("relative energy")
plt.show()

# Part c): one snapshot reconstructed with 1, 2 and 3 modes
snap = 300
plt.plot(xg, Xm[:, snap], color="0.7", linewidth=2.5, label="full field")
for k in (1, 2, 3):
    plt.plot(xg, (U[:, :k] * S[:k]) @ Vt[:k, snap], label=f"{k} mode(s)")
plt.xlabel("$x$")
plt.legend(frameon=False)
plt.show()

## Exercise: controlling the logistic map

Full text: [parts a)–c) in the chapter](https://mvreeuwijk.github.io/chaosbook/python/practice.html#exercises).

In [ ]:
# Parts a)-b): the OGY idea. drmax is the largest allowed nudge of r.
def control(r=3.9, drmax=0.05, n_on=100, N=400, x=0.2):
    xstar = 1 - 1 / r
    orbit = [x]
    for n in range(1, N + 1):
        rr = r
        if n > n_on:                               # the controller is switched on
            dr = xstar / (x * (1 - x)) - r         # the nudge that would hit x*
            if abs(dr) <= drmax:
                rr = r + dr
        x = cb.logistic(x, rr)
        orbit.append(x)
    return xstar, np.array(orbit)

xstar, X = control()
print("x* =", xstar, "  slope of the map at x*:", 3.9 * (1 - 2 * xstar))   # a): unstable?
plt.plot(X, ".-", markersize=3, linewidth=0.5)
plt.axhline(xstar, color="b", ls="--")
plt.axvline(100, color="orange")
plt.xlabel("$n$")
plt.ylabel("$x_n$")
plt.show()

In [ ]:
# Part c): how long does capture take as a function of drmax?
for drmax in (0.1, 0.05, 0.02, 0.01, 0.005, 0.001):
    # average over several initial conditions, since capture time is random
    caps = []
    for x0_ in np.linspace(0.1, 0.9, 30):
        _, X = control(drmax=drmax, N=5000, x=x0_)
        hit = np.nonzero(np.abs(X[101:] - xstar) < 1e-9)[0]
        caps.append(hit[0] if hit.size else np.nan)
    print(f"drmax = {drmax:6.3f}: mean capture time {np.nanmean(caps):8.1f} steps "
          f"({np.sum(np.isnan(caps))} of 30 not captured)")

## Exercise: reconstructing an attractor from one signal

Full text: [parts a)–d) in the chapter](https://mvreeuwijk.github.io/chaosbook/python/practice.html#exercises).

In [ ]:
# Part a): keep only x(t) and plot the delay vectors (x_t, x_{t+tau}, x_{t+2 tau}).
#   Try tau = 3, 10, 18, 40 (in samples of 0.01 time units).
sol = solve_ivp(cb.lorenz, [0, 130], [1, 1, 1], t_eval=np.arange(50, 130, 0.01),
                rtol=1e-9, atol=1e-11)
X = sol.y[0]

def delay_plot(X, tau, ax=None):
    N = len(X) - 2 * tau
    emb = np.column_stack([X[:N], X[tau:N + tau], X[2 * tau:N + 2 * tau]])
    ax = ax or plt.figure(figsize=(5, 5)).add_subplot(projection="3d")
    ax.plot(*emb.T, linewidth=0.3)
    return emb

emb = delay_plot(X, tau=18)
plt.show()

In [ ]:
# Part b): correlation dimension C(eps) ~ eps^D of the reconstruction.
#   Compare with the same estimate for the full (x, y, z) trajectory.
from scipy.spatial.distance import pdist

def correlation_dimension(P, n_pts=1500, seed=0):
    P = P[np.random.default_rng(seed).choice(len(P), n_pts, replace=False)]
    d = pdist(P)
    eps = np.logspace(np.log10(d.max()) - 2.2, np.log10(d.max()) - 0.8, 12)
    C = np.array([(d < e).mean() for e in eps])
    slope = np.polyfit(np.log(eps), np.log(C), 1)[0]
    return slope

print("reconstructed:", correlation_dimension(emb))
print("full (x,y,z) :", correlation_dimension(sol.y.T))

In [ ]:
# Parts c)-d): data you did not generate. Vary dj to get the best picture.
#   dataset2.txt has noise superimposed.
xx = np.loadtxt("data/dataset1.txt")      # swap in "data/dataset2.txt" for part d)
dj = 5
delay_plot(xx, dj)
plt.show()

---
*Back to [the chapter](https://mvreeuwijk.github.io/chaosbook/python/practice.html) — or open the
[cookbook](cookbook.ipynb) in this same lab environment.*